# 06 Data Labeling and Preprocessing

## 📌 Objective  
In this notebook, we will:  
✔ Add **product category labels** to `X_train.pkl`. </br> 
✔ Encode the **`prdtypecode`** column into numeric values (0 to 26).  
✔ Split the dataset for different tasks: **Text (Machine Learning and RNN) and later for Image (CNN - Deep Learning).**  
✔ Save the processed datasets for the next steps.  



## 1. Import Required Libraries 

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import sys
import os
from pathlib import Path
import importlib
import pandas as pd  
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split


### Setting Up Project Paths and Configurations

In [2]:
# Get the current notebook directory
CURRENT_DIR = Path(os.getcwd()).resolve()

# Automatically find the project root (go up 1 level)
PROJECT_ROOT = CURRENT_DIR.parents[1]

# Add project root to sys.path
sys.path.append(str(PROJECT_ROOT))

# Function to get relative paths from project root
def get_relative_path(absolute_path):
    return str(Path(absolute_path).relative_to(PROJECT_ROOT))

# Print project root directory
print(f"Project Root Directory: {PROJECT_ROOT.name}")  # Display only the root folder name

import config  # Now Python can find config.py

Project Root Directory: Data_Scientist_Rakuten_Project-main


## 2. Loading  Data  

We load:  
- **`X_train.pkl`** → Training dataset (without labels).  
- **`y_train.pkl`** → Target variable (before encoding).  
- **`prdtypecode_categories_mapping.pkl`** → Mapping of `prdtypecode` to human-readable labels.  


In [3]:
import importlib
import pickle
import pandas as pd
from pathlib import Path

# Reload config to ensure any updates are applied
importlib.reload(config)  

# Define paths for datasets
train_pickle_path = Path(config.INTERIM_DIR) / "X_train_cleaned.pkl"
test_pickle_path = Path(config.INTERIM_DIR) / "X_test_sub_cleaned.pkl"
y_train_pickle_path = Path(config.INTERIM_DIR) / "y_train.pkl"
mapping_pickle_path = Path(config.PROCESSED_DIR) / "prdtypecode_categories_mapping.pkl"  # Path for category mapping

# Function to get relative paths from project root
def get_relative_path(absolute_path: Path):
    """Returns the relative path from the project root."""
    return str(absolute_path.relative_to(config.BASE_DIR))

# Function to load a Pickle file safely
def load_pickle(file_path: Path, dataset_name: str):
    """Loads a pickle file with error handling and basic visualization."""
    if not file_path.exists():
        print(f"Error: `{dataset_name}` file not found at {file_path}")
        return None

    try:
        data = pd.read_pickle(file_path)
        print(f"Successfully loaded `{dataset_name}` | Type: {type(data)}")

        if isinstance(data, pd.DataFrame) and not data.empty:
            display(data.head())  # Display first rows for DataFrame
        elif isinstance(data, dict) and data:  # If it's a dictionary, show a sample
            sample_items = list(data.items())[:5]
            print(f"Sample entries from `{dataset_name}`: {sample_items}")

        return data
    except Exception as e:
        print(f"Error loading `{dataset_name}`: {e}")
        return None

# List of required files with their names
required_files = {
    "Training Dataset": train_pickle_path,
    "Testing Dataset": test_pickle_path,   
    "Training Labels": y_train_pickle_path,
    "Category Mapping": mapping_pickle_path  
}

# Check if files exist before loading
for name, path in required_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Error: `{name}` file not found at {get_relative_path(path)}")

# Load datasets using the same function
X_train = load_pickle(train_pickle_path, "X_train_cleaned.pkl")
X_test_sub = load_pickle(test_pickle_path, "X_test_sub_cleaned.pkl")
y_train = load_pickle(y_train_pickle_path, "y_train.pkl")
prdtypecode_mapping = load_pickle(mapping_pickle_path, "prdtypecode_categories_mapping.pkl")  # Loaded same way


Successfully loaded `X_train_cleaned.pkl` | Type: <class 'pandas.core.frame.DataFrame'>


,designation,description,productid,imageid,prdtypecode,image_name,text
0,olivia personalisiertes notizbuch seiten punkt...,NaN,3804725264,1263597046,10,image_1263597046_product_3804725264.jpg,olivia personalisiertes notizbuch seiten punkt...
1,journal arts art marche salon art asiatique pa...,NaN,436067568,1008141237,2280,image_1008141237_product_436067568.jpg,journal arts art marche salon asiatique paris ...
2,grand stylet ergonomique bleu gamepad nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,50,image_938777978_product_201115110.jpg,grand stylet ergonomique bleu gamepad nintendo...
3,peluche donald europe disneyland marionnette d...,NaN,50418756,457047496,1280,image_457047496_product_50418756.jpg,peluche donald europe disneyland marionnette d...
4,guerre tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,2705,image_1077757786_product_278535884.jpg,guerre tuques luc idees grandeur veut organise...


Successfully loaded `X_test_sub_cleaned.pkl` | Type: <class 'pandas.core.frame.DataFrame'>


,designation,description,productid,imageid,image_name,text
84916,folkmanis puppets marionnette theatre mini turtle,NaN,516376098,1019294171,image_1019294171_product_516376098.jpg,folkmanis puppets marionnette theatre mini turtle
84917,porte flamme gaxix flamebringer gaxix twilight...,NaN,133389013,1274228667,image_1274228667_product_133389013.jpg,porte flamme gaxix flamebringer twilight dragons
84918,pompe filtration speck badu,NaN,4128438366,1295960357,image_1295960357_product_4128438366.jpg,pompe filtration speck badu
84919,robot piscine electrique,<p>Ce robot de piscine d&#39;un design innovan...,3929899732,1265224052,image_1265224052_product_3929899732.jpg,robot piscine electrique design innovant elega...
84920,hsm destructeur securio coupe croise,NaN,152993898,940543690,image_940543690_product_152993898.jpg,hsm destructeur securio coupe croise


Successfully loaded `y_train.pkl` | Type: <class 'pandas.core.frame.DataFrame'>


,prdtypecode
0,10
1,2280
2,50
3,1280
4,2705


Successfully loaded `prdtypecode_categories_mapping.pkl` | Type: <class 'pandas.core.frame.DataFrame'>


,prdtypecode,Category
0,10,Adult Books
1,40,Imported Video Games
2,50,Video Games Accessories
3,60,Games and Consoles
4,1140,Figurines and Toy Pop


## 3. Adding Product Category Labels  

Now that we have loaded the datasets and the `prdtypecode_categories_mapping.pkl`, we will:  
✔ Add a **new column `Label`** to `X_train`, making product categories more interpretable.  
✔ Ensure proper alignment between `prdtypecode` and category names.  


In [4]:
# Add the category labels to X_train
X_train["Label"] = X_train["prdtypecode"].map(prdtypecode_mapping.set_index("prdtypecode")["Category"])

# Display a sample to verify
print(f"[✔] Added `Label` column to X_train | Shape: {X_train.shape}")
display(X_train.head())


[✔] Added `Label` column to X_train | Shape: (84916, 8)


,designation,description,productid,imageid,prdtypecode,image_name,text,Label
0,olivia personalisiertes notizbuch seiten punkt...,NaN,3804725264,1263597046,10,image_1263597046_product_3804725264.jpg,olivia personalisiertes notizbuch seiten punkt...,Adult Books
1,journal arts art marche salon art asiatique pa...,NaN,436067568,1008141237,2280,image_1008141237_product_436067568.jpg,journal arts art marche salon asiatique paris ...,Magazines
2,grand stylet ergonomique bleu gamepad nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,50,image_938777978_product_201115110.jpg,grand stylet ergonomique bleu gamepad nintendo...,Video Games Accessories
3,peluche donald europe disneyland marionnette d...,NaN,50418756,457047496,1280,image_457047496_product_50418756.jpg,peluche donald europe disneyland marionnette d...,Toys for Children
4,guerre tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,2705,image_1077757786_product_278535884.jpg,guerre tuques luc idees grandeur veut organise...,Books


##  4. Encoding Target Variable (`prdtypecode`) and Reordering Columns

To ensure compatibility with classification models, we perform the following transformations:  
✔ **Convert `prdtypecode` into a numerical format (0-26)** for structured model training.  
✔ **Reorder columns in `X_train` and `X_test_sub`** to maintain a logical and standardized structure.  



### 4.1 Convert prdtypecode into a numerical format (0-26)

In [5]:
# Convert product codes to numerical labels


# Convert product codes to numerical labels
prdtypecodes = sorted(X_train["prdtypecode"].unique())  # Ensure consistent order
target_mapping = {code: i for i, code in enumerate(prdtypecodes)}  # Mapping {prdtypecode: numeric_label}

# print("target_mapping")
# display(target_mapping)

# print("X_train")
# display(X_train)

# display(target_mapping)
        
# Apply mapping to encode target variable
y_train_encoded = y_train["prdtypecode"].map(target_mapping).astype("int8")  # Memory optimization

# Apply encoding to X_train
X_train["prdtypecode_encoded"] = X_train["prdtypecode"].map(target_mapping).astype("int8")


# Display comparison between original and encoded labels
comparison_df = pd.DataFrame({
    "Original prdtypecode": y_train["prdtypecode"].head(10),
    "Encoded prdtypecode (0-26)": y_train_encoded.head(10)
})

print("y_train - Comparison: Original vs. Encoded Labels")
display(comparison_df)

comparison_df_X_train = X_train[["prdtypecode", "prdtypecode_encoded"]].head(10)
print("X_train - Comparison: Original vs. Encoded Labels")
display(comparison_df_X_train)

# print("target_mapping")
# display(target_mapping)

# Final construction of mapping_df
mapping_df = pd.DataFrame({
    "Original prdtypecode": prdtypecodes,
    "Encoded target": [target_mapping[code] for code in prdtypecodes],
    "Label": [X_train[X_train["prdtypecode"] == code]["Label"].iloc[0] for code in prdtypecodes]  # Assuming you have a "Label" column in X_train
})

print("Mapping DataFrame (Original, Encoded, Label):")
display(mapping_df)


y_train - Comparison: Original vs. Encoded Labels


,Original prdtypecode,Encoded prdtypecode (0-26)
0,10,0
1,2280,18
2,50,2
3,1280,7
4,2705,25
5,2280,18
6,10,0
7,2522,21
8,1280,7
9,2582,22


X_train - Comparison: Original vs. Encoded Labels


,prdtypecode,prdtypecode_encoded
0,10,0
1,2280,18
2,50,2
3,1280,7
4,2705,25
5,2280,18
6,10,0
7,2522,21
8,1280,7
9,2582,22


Mapping DataFrame (Original, Encoded, Label):


,Original prdtypecode,Encoded target,Label
0,10,0,Adult Books
1,40,1,Imported Video Games
2,50,2,Video Games Accessories
3,60,3,Games and Consoles
4,1140,4,Figurines and Toy Pop
5,1160,5,Playing Cards
6,1180,6,"Figurines, Masks, and Role-Playing Games"
7,1280,7,Toys for Children
8,1281,8,Board Games
9,1300,9,Remote Controlled Models


### 4.2 Reordering Columns for `X_train` and `X_test_sub`  

In [6]:
# Reorder columns in X_train
X_train = X_train[[
    "designation", "description", "text", "productid", "imageid", 
    "prdtypecode", "prdtypecode_encoded", "Label", "image_name"
]]

# Reorder columns in X_test_sub for consistency (no prdtypecode column)
X_test_sub = X_test_sub[[
    "designation", "description", "text", "productid", "imageid", "image_name"
]]

# Display first few rows to verify changes
print("X_train - Sample after reordering columns:")
display(X_train.head())

print("\n X_test_sub - Sample after reordering columns:")
display(X_test_sub.head())


X_train - Sample after reordering columns:


,designation,description,text,productid,imageid,prdtypecode,prdtypecode_encoded,Label,image_name
0,olivia personalisiertes notizbuch seiten punkt...,NaN,olivia personalisiertes notizbuch seiten punkt...,3804725264,1263597046,10,0,Adult Books,image_1263597046_product_3804725264.jpg
1,journal arts art marche salon art asiatique pa...,NaN,journal arts art marche salon asiatique paris ...,436067568,1008141237,2280,18,Magazines,image_1008141237_product_436067568.jpg
2,grand stylet ergonomique bleu gamepad nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,grand stylet ergonomique bleu gamepad nintendo...,201115110,938777978,50,2,Video Games Accessories,image_938777978_product_201115110.jpg
3,peluche donald europe disneyland marionnette d...,NaN,peluche donald europe disneyland marionnette d...,50418756,457047496,1280,7,Toys for Children,image_457047496_product_50418756.jpg
4,guerre tuques,Luc a des id&eacute;es de grandeur. Il veut or...,guerre tuques luc idees grandeur veut organise...,278535884,1077757786,2705,25,Books,image_1077757786_product_278535884.jpg



 X_test_sub - Sample after reordering columns:


,designation,description,text,productid,imageid,image_name
84916,folkmanis puppets marionnette theatre mini turtle,NaN,folkmanis puppets marionnette theatre mini turtle,516376098,1019294171,image_1019294171_product_516376098.jpg
84917,porte flamme gaxix flamebringer gaxix twilight...,NaN,porte flamme gaxix flamebringer twilight dragons,133389013,1274228667,image_1274228667_product_133389013.jpg
84918,pompe filtration speck badu,NaN,pompe filtration speck badu,4128438366,1295960357,image_1295960357_product_4128438366.jpg
84919,robot piscine electrique,<p>Ce robot de piscine d&#39;un design innovan...,robot piscine electrique design innovant elega...,3929899732,1265224052,image_1265224052_product_3929899732.jpg
84920,hsm destructeur securio coupe croise,NaN,hsm destructeur securio coupe croise,152993898,940543690,image_940543690_product_152993898.jpg


## 5. Splitting Data for Model Training

To ensure a robust evaluation and prevent data leakage, we split `X_train` into **training and validation sets**.  
This will be used for both **Machine Learning and Deep Learning models**.  

###  **Key Considerations for the Split**
✔ **Stratified Split** → Ensures that all categories (`prdtypecode_encoded`) are proportionally represented.  
✔ **80/20 Ratio** → 80% for training, 20% for validation.  
✔ **Same Split for All Models** → Avoids inconsistencies between ML and DL models.  

 **This step ensures that all feature columns (`designation` ,`description` `text`, `image_name`, etc.) and target (`prdtypecode_encoded`) are split together for consistency.**

In [7]:
# Import train_test_split
from sklearn.model_selection import train_test_split

# Define test size and random state for reproducibility
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Split the dataset into training and validation sets (stratified on prdtypecode_encoded)
X_train_split, X_val_split = train_test_split(
    X_train, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=X_train["prdtypecode_encoded"]
)

# Display the shape of the new splits
print(f"[✔] Training set shape: {X_train_split.shape}")
print(f"[✔] Validation set shape: {X_val_split.shape}")

# Check the distribution of target classes in both sets
print("\n Class Distribution in Training Set:")
display(X_train_split["prdtypecode_encoded"].value_counts(normalize=True).sort_index())

print("\n Class Distribution in Validation Set:")
display(X_val_split["prdtypecode_encoded"].value_counts(normalize=True).sort_index())


[✔] Training set shape: (67932, 9)
[✔] Validation set shape: (16984, 9)

 Class Distribution in Training Set:


0     0.036698
1     0.029530
2     0.019799
3     0.009804
4     0.031458
5     0.046547
6     0.008994
7     0.057351
8     0.024377
9     0.059412
10    0.009510
11    0.029338
12    0.038171
13    0.059736
14    0.050668
15    0.009451
16    0.058794
17    0.009701
18    0.056056
19    0.056218
20    0.016737
21    0.058750
22    0.030486
23    0.120223
24    0.029397
25    0.032518
26    0.010275
Name: prdtypecode_encoded, dtype: float64


 Class Distribution in Validation Set:


0     0.036682
1     0.029557
2     0.019783
3     0.009774
4     0.031441
5     0.046573
6     0.009008
7     0.057348
8     0.024376
9     0.059409
10    0.009480
11    0.029322
12    0.038154
13    0.059762
14    0.050695
15    0.009480
16    0.058820
17    0.009715
18    0.056053
19    0.056229
20    0.016722
21    0.058761
22    0.030499
23    0.120231
24    0.029381
25    0.032501
26    0.010245
Name: prdtypecode_encoded, dtype: float64

 ### Visualizing Train-Test Split Samples

In [8]:
# Display a random sample from the training set
print("\n Random Sample from X_train_split:")
display(X_train_split.sample(4, random_state=42))

# Display a random sample from the validation set
print("\n Random Sample from X_val_split:")
display(X_val_split.sample(4, random_state=42))



 Random Sample from X_train_split:


,designation,description,text,productid,imageid,prdtypecode,prdtypecode_encoded,Label,image_name
8505,retrobit sega mega drive button usb sega megad...,NaN,retrobit sega mega drive button usb megadrive ...,4023069514,1277391030,60,3,Games and Consoles,image_1277391030_product_4023069514.jpg
35442,linxor bache bulles rectangle microns,NaN,linxor bache bulles rectangle microns,2059868714,1123660178,2583,23,Piscine and Spa,image_1123660178_product_2059868714.jpg
70129,pcs mah batterie lithium pieces rechange hubsa...,NaN,pcs mah batterie lithium pieces rechange hubsa...,4042485063,1280126645,1300,9,Remote Controlled Models,image_1280126645_product_4042485063.jpg
13617,dolce gusto latte macchiato vanilla lot capsul...,Nescafé Dolce Gusto Latte Macchiato Vanilla 16...,dolce gusto latte macchiato vanilla lot capsul...,4056001974,1282294376,1940,15,Food,image_1282294376_product_4056001974.jpg



 Random Sample from X_val_split:


,designation,description,text,productid,imageid,prdtypecode,prdtypecode_encoded,Label,image_name
33497,bache blanche plastique grs bache toile moeill...,Bâche blanche plastique 90grs bache Toile 4mx6...,bache blanche plastique grs toile moeillets me...,2064837612,1123909321,2582,22,"Furniture, Kitchen, and Garden",image_1123909321_product_2064837612.jpg
34500,sbad demon jinn exorciste rituels commune,NaN,sbad demon jinn exorciste rituels commune,3972330738,1270553943,1160,5,Playing Cards,image_1270553943_product_3972330738.jpg
28151,illustration,LES FETES DE LISIEUX / LE LEGAT PONTIFICAL PRO...,illustration fetes lisieux legat pontifical pr...,110678737,879650583,2280,18,Magazines,image_879650583_product_110678737.jpg
1018,biberon dur nourrisseur bebe mois rose,Caractéristiques:<br />Caractéristique du maté...,biberon dur nourrisseur bebe mois rose caracte...,3376295323,1209388779,1320,12,Early Childhood,image_1209388779_product_3376295323.jpg


## 6. Saving Data for Future Use

Now that we have **added category labels, encoded the target variable, and split the dataset**, we save the processed files for future use.  

### 🔹 **Saved Files & Their Purpose**

✔ **`X_train_cleaned_final.pkl`** → Full training dataset **after adding labels and encoding** (before split).  
✔ **`X_train_split.pkl`** → Training dataset after preprocessing (80% of the data).  
✔ **`X_val_split.pkl`** → Validation dataset after preprocessing (20% of the data).  
✔ **`prdtypecode_mapping.pkl`** → Mapping of `prdtypecode` to numeric labels (0-26), ensuring consistency across models.
✔ **`X_test_sub_cleaned_final.pkl`** → Test dataset (for submission) after column reordering.  

These files will be used in the upcoming notebooks for **text vectorization, ML models, and deep learning.**




In [9]:
# Define save paths in the processed directory
save_dir = Path(config.PROCESSED_DIR)
save_dir.mkdir(parents=True, exist_ok=True)  # Ensure directory exists

# Paths for saving the processed datasets
full_train_path = save_dir / "X_train_cleaned_final.pkl"
train_split_path = save_dir / "X_train_split.pkl"
val_split_path = save_dir / "X_val_split.pkl"
test_sub_path = save_dir / "X_test_sub_cleaned_final.pkl"  # NEW FILE
mapping_path = save_dir / "prdtypecode_mapping.pkl"

# Save full X_train (after adding labels and encoding)
X_train.to_pickle(full_train_path)

# Save training and validation sets
X_train_split.to_pickle(train_split_path)
X_val_split.to_pickle(val_split_path)

# Save X_test_sub after reordering columns
X_test_sub.to_pickle(test_sub_path)

# # Save the prdtypecode mapping
# with open(mapping_path, "wb") as f:
#     pickle.dump(target_mapping, f)
    
    
# Save X_test_sub after reordering columns
mapping_df.to_pickle(mapping_path)

print(f"[✔] Full cleaned final training dataset saved at: {full_train_path}")
print(f"[✔] Training set saved at: {train_split_path}")
print(f"[✔] Validation set saved at: {val_split_path}")
print(f"[✔] Test dataset saved at: {test_sub_path}")
print(f"[✔] prdtypecode mapping saved at: {mapping_path}")


[✔] Full cleaned final training dataset saved at: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\data\processed\X_train_cleaned_final.pkl
[✔] Training set saved at: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\data\processed\X_train_split.pkl
[✔] Validation set saved at: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\data\processed\X_val_split.pkl
[✔] Test dataset saved at: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\data\processed\X_test_sub_cleaned_final.pkl
[✔] prdtypecode mapping saved at: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\data\processed\prdtypecode_mapping.pkl


## 6. 🔄 Next Steps

Our dataset is now **fully preprocessed** and ready for modeling.  
The next step is to **prepare the text data** for Machine Learning.  

📌 **In the next notebook (`07_ML_Text_Vectorization_TF-IDF.ipynb`)**, we will:  
✔ Extract the text features from **`X_train_split.pkl`**.  
✔ Convert them into a numerical format using **TF-IDF Vectorization**.  
✔ Get the dataset ready for model training.  

**This transformation will allow Machine Learning models to process and analyze textual data.**  
